<a href="https://colab.research.google.com/github/edwardsnj/pride-study-retrieval-cofest-2026/blob/main/notebooks/TextEmbedding%2BTFIDF_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#
# Import necessary modules
#

import sys, os, httpimport

with httpimport.github_repo("EdwardsLabProjects", "pride-study-retrieval-cofest-2026", ref="main"):
    from notebooks.lib import *

set_random_seed(state=None)

Version: 1.0.26
Using random seed: 6603116


In [2]:
#
# Get embeddings and known true positives
#

md,emb = embeddings("openai-3-small")
embdim = emb.shape[0]
tp,tn = knownstudies()

allacc = set(emb.columns)
tp &= allacc
tn &= allacc

assert len(tp & tn) == 0, "TP and TN should not intersect!"

# Print out some details...
print(f"Embeddings: {emb.shape}")
print(f"True Pos: {len(tp)}")
print(f"True Neg: {len(tn)}")

allacc,avgemb = select_by_embedding_proximity(tp,emb,1000)
print(f"All Acc: {len(allacc)}")

Embeddings: (1536, 36696)
True Pos: 84
True Neg: 47
All Acc: 1000


In [3]:
#
# Create train/test split
#
TESTFRAC = 0.2
BACKGROUND_OVERSAMPLE = 25

train_acc, train_y, test_acc, test_y = split_train_test(allacc, tp, tn,
    test_size=TESTFRAC, bgsize=BACKGROUND_OVERSAMPLE)


In [4]:
# Build the TF-IDF features from train data only,
# but build feature vectors for all documents

from sklearn.feature_extraction import text
stop_words = list(set(list(text.ENGLISH_STOP_WORDS) + """

""".split()))

tfidf_extra_args = dict(positive_only=False, # Use only positive set for word selection
                        min_df=1, # minimum documents word is in
                        max_df=1.0, # maximum documents word is in
                        max_features=None, # max number of words to use
                        stop_words=stop_words) # words to ignore

tfidf,tfidf_model = create_tfidf_features(md, train_acc, train_y,
                                          test_acc, **tfidf_extra_args)

print(f"TF-IDF features shape: {tfidf.shape}")

TF-IDF features shape: (21448, 1000)


In [5]:
logreg_extra_args = dict(class_weight='balanced',C=10.0,
                         penalty='l1', solver='liblinear',
                         use_embed=True, use_tfidf=True)
trained_model = train_document_classifier(emb, tfidf,
                                          train_acc, train_y,
                                          test_acc, test_y,
                                          **logreg_extra_args)
print("NZ Coeffs:",sum(*[1*(xi!=0) for xi in trained_model.coef_]))

Training data shape: (800, 22984) (Positives: 67, Negatives: 733)
Testing data shape: (200, 22984) (Positives: 17, Negatives: 183)

--- Model Evaluation ---
Accuracy: 0.9400

Classification Report:
                precision    recall  f1-score   support

Background (0)       0.97      0.97      0.97       183
 Seed-like (1)       0.65      0.65      0.65        17

      accuracy                           0.94       200
     macro avg       0.81      0.81      0.81       200
  weighted avg       0.94      0.94      0.94       200

NZ Coeffs: 80


In [6]:
nzemb,nztfidf,topfeat = top_features(trained_model,tfidf_model,nembed=embdim,**logreg_extra_args)
print(f"Number of sem. embedding dimensions with non-zero coefficients: {nzemb}")
print(f"Number of TF-IDF dimensions with non-zero coefficients: {nztfidf}")
print("\nTop 20 TF-IDF Features:")
display(topfeat.head(20))

show_top_feature_examples(topfeat, md)


Number of sem. embedding dimensions with non-zero coefficients: 8
Number of TF-IDF dimensions with non-zero coefficients: 72

Top 20 TF-IDF Features:


,Feature,Coefficient
13939,neb,34.100902
9620,glycosylated,32.108061
6744,deamidation,30.466204
355,106,30.197320
924,18o,29.458548
15751,pngase,23.659568
9607,glycosite,20.286166
10611,hydrazide,20.164628
6237,contortus,-17.605858
4962,bx,17.155649



=== 'neb' (coef: +34.1009) ===
  PXD001432: The beads were washed with 40 µl 1 x G7 buffer (NEB, Frankfurt a.M., Germany) and then incubated with 20 µl 1x G7 buffer containing 500 U glyerol-free PNGase F (NEB) at 37 °C in a thermomixer for 6 h to release the glycopeptides from the beads.
  PXD010911: First, 2 µL GlycoBuffer 2 (NEB, 50 mM sodium phosphate, pH 7.5) was aliquoted and dried under vacuum; next, 20 μL of H218O (99% 18O, Cambridge Isotopes, Andover, MA) was added, mixed, and transferred to the aliquot of immunoglobulin peptides.
  PXD001034: All MS/MS data were analyzed using Phenyx (GeneBio, Geneva, Switzerland) and X!Tandem (The GPM, thegpm.org; version CYCLONE (2010.12.01.1)).
  PXD001506: Data were acquired using an ion spray voltage of 2.4kV, curtain gaz of 30 PSI, nebulizer gaz of 8 PSI an an interface heater temperature of 125C.
  PXD002855: Re-calibrated mass lists were searched using Mascot and Phenyx (GeneBio, SA, version 2.5.14) database search algorithms with mas